# IEP-1001 CASE 3 — RAGAS 4대 지표 평가

**목적**: 배포용으로 채택한 CASE 3의 Context Precision / Faithfulness / Answer Relevancy 추가 측정  
**컬렉션**: `ipcc_1001_case3_v1` (506개 청크, chunk_size=1000, overlap=200)  
**Judge LLM**: `llama3.1:8b`  
**임베딩 모델**: `jhgan/ko-sroberta-multitask`  
**대상 문서**: IPCC AR6 SYR 한글 번역본 (188페이지)

| 지표 | 구분 | CASE 3 (기존) | IEP-1002 | IEP-1003* |
| :--- | :--- | :---: | :---: | :---: |
| Context Recall | Retriever | **0.8520** | 0.7106 | 0.8611 |
| Context Precision | Retriever | 미측정 | 0.2150 | 0.5544 |
| Faithfulness | Generator | 미측정 | 0.2258 | 0.3869 |
| Answer Relevancy | Generator | 미측정 | 0.5100 | 0.5693 |

> *IEP-1003 수치는 NaN 85개 생존 편향 포함 (유효 샘플 15개 기준)

**평가 전략**: T4 메모리 제약으로 2회 분리 실행
- 1회차: Context Recall + Context Precision
- 2회차: Faithfulness + Answer Relevancy

> **사전 준비**: Google Drive에 아래 경로 존재 확인
> - `IPCC_data/IEP_1001_CASE3/chroma_db/` (ipcc_1001_case3_v1, 506개 청크)
> - `IPCC_data/IEP_1002/golden_dataset_100.csv`


## 1. 환경 설정

### 1-1. Config

In [ ]:
# ── 경로 ───────────────────────────────────────────────────────────
CHROMA_DIR  = "/content/drive/MyDrive/IPCC_data/IEP_1001_CASE3/chroma_db"
GOLDEN_CSV  = "/content/drive/MyDrive/IPCC_data/IEP_1002/golden_dataset_100.csv"
RESULTS_DIR = "/content/drive/MyDrive/IPCC_data/IEP_1001_CASE3/results"

# ── 컬렉션 / 모델 ──────────────────────────────────────────────────
COLLECTION_NAME = "ipcc_1001_case3_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"
JUDGE_MODEL     = "llama3.1:8b"
TOP_K           = 3
EXPECTED_CHUNKS = 506

# ── 저장 경로 ──────────────────────────────────────────────────────
SAVE_RETRIEVED = f"{RESULTS_DIR}/iep1001_case3_retrieved.csv"
SAVE_RETRIEVER = f"{RESULTS_DIR}/iep1001_case3_retriever.csv"
SAVE_GENERATOR = f"{RESULTS_DIR}/iep1001_case3_generator.csv"
SAVE_RAW       = f"{RESULTS_DIR}/iep1001_case3_raw.csv"
SAVE_SUMMARY   = f"{RESULTS_DIR}/iep1001_case3_summary.csv"

# ── 비교 기준 (이전 실험 결과) ─────────────────────────────────────
IEP1002 = {
    "전체":     {"context_recall": 0.7106, "context_precision": 0.2150, "faithfulness": 0.2258, "answer_relevancy": 0.5100},
    "사실 확인": {"context_recall": 0.3333, "context_precision": 0.3267, "faithfulness": 0.2794, "answer_relevancy": 0.5130},
    "비교":     {"context_recall": 0.7619, "context_precision": 0.2133, "faithfulness": 0.2143, "answer_relevancy": 0.5073},
    "의견/예측": {"context_recall": 0.9000, "context_precision": 0.3067, "faithfulness": 0.2222, "answer_relevancy": 0.5161},
    "범위 밖":  {"context_recall": 0.7353, "context_precision": 0.0133, "faithfulness": 0.1875, "answer_relevancy": 0.5041},
}
IEP1003_BIASED = {  # NaN 85개 생존 편향 포함 수치
    "전체":     {"context_recall": 0.8611, "context_precision": 0.5544, "faithfulness": 0.3869, "answer_relevancy": 0.5693},
    "사실 확인": {"context_recall": 0.6667, "context_precision": 0.6894, "faithfulness": 0.7500, "answer_relevancy": 0.4875},
    "비교":     {"context_recall": 1.0000, "context_precision": 0.5606, "faithfulness": 0.3333, "answer_relevancy": 0.6661},
    "의견/예측": {"context_recall": 0.7833, "context_precision": 0.8177, "faithfulness": 0.4028, "answer_relevancy": 0.6330},
    "범위 밖":  {"context_recall": 1.0000, "context_precision": 0.0303, "faithfulness": 0.1667, "answer_relevancy": 0.4842},
}

print("✅ Config 완료")
print(f"   컬렉션   : {COLLECTION_NAME} ({EXPECTED_CHUNKS}개)")
print(f"   Judge LLM: {JUDGE_MODEL}")
print(f"   결과 저장 : {RESULTS_DIR}")


### 1-2. 라이브러리 설치

In [ ]:
!pip install -qU chromadb langchain-community langchain-huggingface langchain-ollama ragas tqdm
print("✅ 설치 완료")


### 1-3. Google Drive 마운트

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CHROMA_DIR), f"❌ ChromaDB 없음: {CHROMA_DIR}"
assert os.path.exists(GOLDEN_CSV), f"❌ 골든 데이터셋 없음: {GOLDEN_CSV}"
print("✅ Drive 마운트 완료")
print(f"   ChromaDB   : {CHROMA_DIR}")
print(f"   골든 데이터셋: {GOLDEN_CSV}")


### 1-4. Ollama 설치 및 서버 시작

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama 설치 완료")


In [ ]:
import subprocess, time, requests

def start_ollama():
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for i in range(30):
        try:
            if requests.get("http://localhost:11434", timeout=2).status_code == 200:
                print(f"✅ Ollama 서버 준비 완료 ({i+1}초)")
                return True
        except:
            pass
        time.sleep(1)
    print("❌ 서버 시작 실패")
    return False

def check_ollama():
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("✅ Ollama 서버 정상")
        return True
    except:
        print("❌ 서버 응답 없음 — start_ollama()를 다시 실행하세요.")
        return False

start_ollama()


### 1-5. Ollama 모델 다운로드

In [ ]:
!ollama pull {JUDGE_MODEL}    # Judge LLM
!ollama pull nomic-embed-text  # RAGAS 내부 평가용 임베딩
print("✅ 모델 준비 완료")


## 2. ChromaDB + 임베딩 모델 로드

한국어 특화 임베딩 모델 `jhgan/ko-sroberta-multitask`를 로드하고, CASE 3 컬렉션을 불러온다.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)
count = vectorstore._collection.count()
print(f"✅ ChromaDB 로드: {COLLECTION_NAME} ({count}개 청크)")
assert count == EXPECTED_CHUNKS, f"⚠️  기대값 {EXPECTED_CHUNKS}개, 실제 {count}개 — CHROMA_DIR 경로를 확인하세요."


## 3. 골든 데이터셋 로드

In [ ]:
import pandas as pd

df_gold = pd.read_csv(GOLDEN_CSV)
print(f"✅ {len(df_gold)}개 로드")
print(f"   컬럼명: {df_gold.columns.tolist()}")
assert len(df_gold) == 100, f"⚠️  골든 데이터셋 {len(df_gold)}개 — 100개여야 합니다."
assert 'user_input' in df_gold.columns, f"❌ 'user_input' 컬럼 없음. 실제 컬럼: {df_gold.columns.tolist()}"
assert 'reference'  in df_gold.columns, f"❌ 'reference' 컬럼 없음. 실제 컬럼: {df_gold.columns.tolist()}"
print(df_gold['Type'].value_counts().to_string())


## 4. Retrieval

검색 결과(retrieved_contexts)를 먼저 수집해 중간 저장한다. Answer 생성은 다음 단계에서 별도로 실행한다.

In [ ]:
import subprocess, requests as req, time
from tqdm.notebook import tqdm
import pandas as pd, os

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

records = []
for i, row in tqdm(df_gold.iterrows(), total=len(df_gold), desc="Retrieval"):
    retrieved = retriever.invoke(row['user_input'])
    records.append({
        "id":                 row['ID'],
        "type":               row['Type'],
        "user_input":         row['user_input'],
        "retrieved_contexts": [doc.page_content for doc in retrieved],
        "answer":             "",
        "reference":          row['reference'],
    })

print(f"✅ Retrieval 완료: {len(records)}개")
print(f"   컨텍스트 수 확인: {[len(r['retrieved_contexts']) for r in records[:3]]}")


## 5. Answer 생성

- `num_predict`: 256 (답변 길이 단축으로 생성 속도 약 2배 향상)
- 10개마다 체크포인트 저장 — 세션이 끊겨도 이어서 실행 가능
- `ensure_ollama()` 자동 재시작 포함

In [ ]:
RAG_PROMPT = """다음 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 내용은 보고서에서 찾을 수 없습니다."라고 답하세요.

[문서]
{context}

[질문]
{question}

[답변]"""

def ensure_ollama():
    try:
        req.get("http://localhost:11434", timeout=3)
    except:
        print("  🔄 Ollama 재시작 중...")
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            time.sleep(1)
            try:
                req.get("http://localhost:11434", timeout=2)
                print("  ✅ Ollama 재시작 완료")
                return
            except:
                pass

def generate_answer(question, contexts, retry=2):
    context = "\n\n".join(f"[문서 {i+1}]\n{c}" for i, c in enumerate(contexts))
    for attempt in range(retry + 1):
        ensure_ollama()
        try:
            resp = req.post(
                "http://localhost:11434/api/generate",
                json={
                    "model":       JUDGE_MODEL,
                    "prompt":      RAG_PROMPT.format(context=context, question=question),
                    "stream":      False,
                    "num_predict": 256,
                },
                timeout=180
            )
            return resp.json().get("response", "").strip()
        except Exception as e:
            if attempt == retry:
                return f"ERROR: {e}"
            time.sleep(5)

def save_checkpoint(records, path):
    df = pd.DataFrame([{
        "id": r["id"], "type": r["type"], "user_input": r["user_input"],
        "answer": r["answer"], "reference": r["reference"],
        "ctx_0": r["retrieved_contexts"][0] if len(r["retrieved_contexts"]) > 0 else "",
        "ctx_1": r["retrieved_contexts"][1] if len(r["retrieved_contexts"]) > 1 else "",
        "ctx_2": r["retrieved_contexts"][2] if len(r["retrieved_contexts"]) > 2 else "",
    } for r in records])
    df.to_csv(path, index=False, encoding='utf-8-sig')

# ── Answer 생성: 10개마다 체크포인트 저장 ─────────────────────────
start_idx = sum(1 for r in records if r['answer'] != "")
if start_idx > 0:
    print(f"⏩ {start_idx}개 완료 — {start_idx}번부터 이어서 실행")

for i in tqdm(range(start_idx, len(records)), desc="Answer 생성", initial=start_idx, total=len(records)):
    records[i]['answer'] = generate_answer(
        records[i]['user_input'],
        records[i]['retrieved_contexts']
    )
    if (i + 1) % 10 == 0:
        save_checkpoint(records, SAVE_RETRIEVED)
        print(f"  💾 체크포인트: {i+1}/100")

save_checkpoint(records, SAVE_RETRIEVED)
print(f"\n✅ Answer 생성 완료: {len(records)}개")
print(f"   저장: {SAVE_RETRIEVED}")
error_count = sum(1 for r in records if r['answer'].startswith('ERROR'))
if error_count:
    print(f"   ⚠️  ERROR 발생: {error_count}개 — RAGAS 평가 시 NaN으로 처리됩니다.")


## 6. (선택) 세션 재시작 시 중간 저장 파일 복원

> Step 4~5를 이미 실행했다면 이 셀은 건너뛰세요.

In [ ]:
import ast

df_retrieved = pd.read_csv(SAVE_RETRIEVED)
records = []
for _, row in df_retrieved.iterrows():
    contexts = [row[f"ctx_{i}"] for i in range(3) if pd.notna(row.get(f"ctx_{i}", "")) and row.get(f"ctx_{i}", "") != ""]
    records.append({
        "id":                 row["id"],
        "type":               row["type"],
        "user_input":         row["user_input"],
        "retrieved_contexts": contexts,
        "answer":             row["answer"],
        "reference":          row["reference"],
    })

print(f"✅ 중간 저장 파일 복원 완료: {len(records)}개")


## 7. RAGAS 평가 모델 설정

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset

ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=JUDGE_MODEL, base_url="http://localhost:11434", temperature=0, num_predict=512)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
)
run_config = RunConfig(max_workers=4, timeout=600, max_retries=3)

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":          r['user_input'],
    "retrieved_contexts":  r['retrieved_contexts'],
    "response":            r['answer'],
    "reference":           r['reference'],
} for r in records]))

print(f"✅ 평가 모델 설정 완료 / EvaluationDataset: {len(eval_dataset)}개")


## 8. 1회차 평가: Context Recall + Context Precision

In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision

print(f"▶ 1회차 평가 시작 (judge: {JUDGE_MODEL})")
result_retriever = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_retriever = result_retriever.to_pandas()
df_retriever['id']   = [r['id']   for r in records]
df_retriever['type'] = [r['type'] for r in records]
df_retriever.to_csv(SAVE_RETRIEVER, index=False, encoding='utf-8-sig')

print("\n✅ 1회차 완료")
print(df_retriever[['context_recall','context_precision']].mean().round(4).to_string())
print("NaN:", df_retriever[['context_recall','context_precision']].isna().sum().to_dict())


## 9. 2회차 평가: Faithfulness + Answer Relevancy

> 1회차 완료 후 `check_ollama()`로 서버 상태를 확인한 뒤 진행하세요.

In [ ]:
check_ollama()

In [ ]:
from ragas.metrics import Faithfulness, AnswerRelevancy

print(f"▶ 2회차 평가 시작 (judge: {JUDGE_MODEL})")
result_generator = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_generator = result_generator.to_pandas()
df_generator['id']   = [r['id']   for r in records]
df_generator['type'] = [r['type'] for r in records]
df_generator.to_csv(SAVE_GENERATOR, index=False, encoding='utf-8-sig')

print("\n✅ 2회차 완료")
print(df_generator[['faithfulness','answer_relevancy']].mean().round(4).to_string())
print("NaN:", df_generator[['faithfulness','answer_relevancy']].isna().sum().to_dict())


## 10. 결과 병합 + 유형별 분석

In [ ]:
import numpy as np

METRICS = ['context_recall', 'context_precision', 'faithfulness', 'answer_relevancy']

df_raw = pd.DataFrame({
    'id':                [r['id']   for r in records],
    'type':              [r['type'] for r in records],
    'user_input':        [r['user_input'] for r in records],
    'context_recall':    df_retriever['context_recall'].values,
    'context_precision': df_retriever['context_precision'].values,
    'faithfulness':      df_generator['faithfulness'].values,
    'answer_relevancy':  df_generator['answer_relevancy'].values,
})

overall_mean = df_raw[METRICS].mean().round(4)
overall_nan  = df_raw[METRICS].isna().sum()
summary      = df_raw.groupby('type')[METRICS].mean().round(4)
nan_by_type  = df_raw.groupby('type')[METRICS].apply(lambda x: x.isna().sum())

print("전체 평균 (NaN 제외):")
for m in METRICS:
    print(f"  {m:<22}: {overall_mean[m]:.4f}  (NaN: {overall_nan[m]}개)")
print("\n유형별 평균:")
print(summary.to_string())
print("\n유형별 NaN:")
print(nan_by_type.to_string())


## 11. 3종 비교 + 생존 편향 보정 + 최종 저장

In [ ]:
types_order = ['사실 확인', '비교', '의견/예측', '범위 밖']

# ── 4지표 3종 비교 ────────────────────────────────────────────────
print("=" * 80)
print("  4지표 비교: IEP-1001 CASE3 vs IEP-1002 vs IEP-1003*")
print("  *IEP-1003은 NaN 85개 생존 편향 포함 수치")
print("=" * 80)

for metric in METRICS:
    print(f"\n[{metric}]")
    print(f"  {'유형':<10} {'CASE3':>12} {'IEP-1002':>10} {'IEP-1003*':>10} {'vs 1002':>8} {'vs 1003':>8}")
    print("  " + "-" * 62)
    for t in types_order:
        n  = summary.loc[t, metric] if t in summary.index else np.nan
        b2 = IEP1002.get(t, {}).get(metric, np.nan)
        b3 = IEP1003_BIASED.get(t, {}).get(metric, np.nan)
        d2 = f"{n - b2:+.4f}" if not (np.isnan(n) or np.isnan(b2)) else "   N/A"
        d3 = f"{n - b3:+.4f}" if not (np.isnan(n) or np.isnan(b3)) else "   N/A"
        print(f"  {t:<10} {n:>12.4f} {b2:>10.4f} {b3:>10.4f} {d2:>8} {d3:>8}")
    n_all  = overall_mean[metric]
    b2_all = IEP1002['전체'][metric]
    b3_all = IEP1003_BIASED['전체'][metric]
    print("  " + "-" * 62)
    print(f"  {'전체':<10} {n_all:>12.4f} {b2_all:>10.4f} {b3_all:>10.4f} {n_all-b2_all:>+8.4f} {n_all-b3_all:>+8.4f}")


In [ ]:
# ── 생존 편향 보정: NaN → 0점 처리 ─────────────────────────────
print("\n" + "=" * 60)
print("  생존 편향 보정 (NaN → 0점 처리, 전체 100개 기준)")
print("=" * 60)
print(f"  {'지표':<22} {'낙관적(NaN제외)':>15} {'유효샘플':>8} {'보수적(전체100개)':>18}")
print("  " + "-" * 65)
for m in METRICS:
    optimistic  = overall_mean[m]
    valid_n     = 100 - overall_nan[m]
    pessimistic = round(optimistic * valid_n / 100, 4)
    print(f"  {m:<22} {optimistic:>15.4f} {valid_n:>8}개 {pessimistic:>18.4f}")
print()
print("  Context Recall 기준 비교 (IEP-1001 CASE 3 배포 채택 근거)")
case3_recall_pessimistic = round(overall_mean['context_recall'] * (100 - overall_nan['context_recall']) / 100, 4)
print(f"  IEP-1001 CASE3 보수적 Recall: {case3_recall_pessimistic:.4f}")
print(f"  IEP-1003 보수적 Recall       : {round(0.8611 * 15 / 100, 4):.4f}  (15개 유효)")


In [ ]:
# ── IEP 문서 / README 용 마크다운 출력 ─────────────────────────
print("\n[IEP-1001 문서 및 README 마크다운]")
md_table = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md_table += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in types_order:
    if t in summary.index:
        r = summary.loc[t]
        md_table += f"| {t} | {r['context_recall']:.4f} | {r['context_precision']:.4f} | {r['faithfulness']:.4f} | {r['answer_relevancy']:.4f} |\n"
md_table += f"| **전체** | **{overall_mean['context_recall']:.4f}** | **{overall_mean['context_precision']:.4f}** | **{overall_mean['faithfulness']:.4f}** | **{overall_mean['answer_relevancy']:.4f}** |\n"
print(md_table)

nan_md = "| 지표 | 낙관적 (NaN 제외) | 유효 샘플 | 보수적 (전체 100개) |\n"
nan_md += "| :--- | :---: | :---: | :---: |\n"
for m in METRICS:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    pes = round(opt * vn / 100, 4)
    nan_md += f"| {m} | {opt:.4f} | {vn}개 | **{pes:.4f}** |\n"
print("[생존 편향 보정 마크다운]")
print(nan_md)


In [ ]:
# ── 최종 저장 ────────────────────────────────────────────────────
df_raw.to_csv(SAVE_RAW, index=False, encoding='utf-8-sig')
df_s = summary.copy()
df_s.loc['전체'] = overall_mean
df_s.to_csv(SAVE_SUMMARY, encoding='utf-8-sig')

print("✅ 저장 완료")
print(f"  raw     → {SAVE_RAW}")
print(f"  summary → {SAVE_SUMMARY}")
print(f"""
╔══════════════════════════════════════════════════════════╗
  IEP-1001 CASE 3 RAGAS 4지표 평가 완료
╚══════════════════════════════════════════════════════════╝

■ 전체 평균 (NaN 제외 낙관적 수치)
  Context Recall    : {overall_mean['context_recall']:.4f}  (NaN: {overall_nan['context_recall']}개)
  Context Precision : {overall_mean['context_precision']:.4f}  (NaN: {overall_nan['context_precision']}개)
  Faithfulness      : {overall_mean['faithfulness']:.4f}  (NaN: {overall_nan['faithfulness']}개)
  Answer Relevancy  : {overall_mean['answer_relevancy']:.4f}  (NaN: {overall_nan['answer_relevancy']}개)

■ IEP-1003 대비
  NaN 기준 CASE3 안정성: {100 - overall_nan['context_recall']}개 유효 (IEP-1003: 15개)
""")
